In [24]:
import sqlite3
import pandas as pd

conn=sqlite3.connect(":memory:")
cur=conn.cursor()



print("SQLite is ready.")


SQLite is ready.


In [25]:
import sqlite3
import pandas as pd

conn=sqlite3.connect(":memory:")
cur=conn.cursor()

cur.execute("""
 Create table employees (
    emp_id Integer primary key,
    emp_name text,
    department text,
    current_location text,
    level text
);
 """)
conn.commit()
print("Table created successfully.")


Table created successfully.


In [26]:
pd.read_sql_query("select * from employees;",conn)


,emp_id,emp_name,department,current_location,level


In [27]:
for emp_id in range(1, 61):
    emp_name = f"Emp{emp_id:03d}"

    mod = emp_id % 6
    if mod == 0:
        department = "Operations"
    elif mod == 1:
        department = "IT"
    elif mod == 2:
        department = "HR"
    elif mod == 3:
        department = "Finance"
    elif mod == 4:
        department = "Customer Support"
    else:
        department = "Security"

    if emp_id <= 45:
        current_location = "HQ"
    else:
        current_location = "Site_B"

    if emp_id <= 10:
        level = "Lead"
    elif emp_id <= 30:
        level = "Senior"
    else:
        level = "Associate"

    cur.execute(
        "INSERT INTO employees (emp_id, emp_name, department, current_location, level) VALUES (?, ?, ?, ?, ?)",
        (emp_id, emp_name, department, current_location, level)
    )

conn.commit()

print("60 employees inserted.")

60 employees inserted.


In [28]:
pd.read_sql_query("select count(*)as total_employees from employees",conn)

,total_employees
0,60


In [29]:
pd.read_sql_query("""
select department,count(*) as headcount from employees group by department order by headcount desc, department""",conn)

,department,headcount
0,Customer Support,10
1,Finance,10
2,HR,10
3,IT,10
4,Operations,10
5,Security,10


In [30]:
staffing_plan = pd.read_sql_query("""
SELECT
    emp_id,
    emp_name,
    department,
    current_location,
    level,

    CASE
        WHEN department IN ('IT','Security') THEN 'Austin'
        WHEN department = 'Finance' THEN 'Chicago'
        WHEN department = 'HR' THEN 'Atlanta'
        ELSE current_location
    END AS new_location,

    CASE
        WHEN department IN ('IT','Security','Finance','HR') THEN 'Relocate'
        ELSE 'Stay'
    END AS move_status

FROM employees
""", conn)

staffing_plan.head()

,emp_id,emp_name,department,current_location,level,new_location,move_status
0,1,Emp001,IT,HQ,Lead,Austin,Relocate
1,2,Emp002,HR,HQ,Lead,Atlanta,Relocate
2,3,Emp003,Finance,HQ,Lead,Chicago,Relocate
3,4,Emp004,Customer Support,HQ,Lead,HQ,Stay
4,5,Emp005,Security,HQ,Lead,Austin,Relocate


In [31]:
relocation_summary = pd.read_sql_query("""
select move_status,
count(*) as total_employees
from (
  select
  case
  when department in
('IT', 'Security', 'Finance','HR') Then 'Relocate'
  else 'Stay'
 End as move_status
 from employees
 )
 group by move_status
 """,conn)
relocation_summary

,move_status,total_employees
0,Relocate,40
1,Stay,20


In [32]:
location_summary=(staffing_plan.groupby('current_location').size().reset_index(name='headcount'))

location_summary


,current_location,headcount
0,HQ,45
1,Site_B,15


This project analyzes employee distribution across departments and relocation patterns using Sqlite and Python. The goal is to understand how employees are distributed across locations and how relocation impacts staffing levels.

Tools Used


*   SQLite
*   Python

Dataset

This Dataset represents employee records, including:



*   Employee ID
*   Employee Name
*   Department
*   Current Location
*   Employee Level

The Data was used to simulate a workforce relocation scenario.

Key Questions
Which locations have the highest employee headcount after relocation?

How are employees distributed across departments?

Which locations are gaining or losing employees?

What trends can be observed in workforce movement?

Analysis
The project uses SQL queries within a SQLite database to:


*   Insert and structure employee data
*   Group employees by department and location
*   Calculate headcounts and distrubutions
*   Analyze relocation patterns


















In [34]:
new_location_summary=(staffing_plan.groupby('new_location').size().reset_index(name='headcount'))

new_location_summary

,new_location,headcount
0,Atlanta,10
1,Austin,20
2,Chicago,10
3,HQ,14
4,Site_B,6


Key Findings:



*   Austin has the highest projected headcount Twenty employees, indicating it will serve as a primary growth hub for the organization.

*   Atlanta and Chicago each have 10 employees suggesting a balanced distribution of staff across these regional locations.

* Headquarters retains 14 employees maintaining a strong central presence to support operations and leadership functions.  

* Site B has the smallest headcount 6 employees. Indicating it is either a newly developing location or a lower priority operational site.

* Overall, 40 employees are being relocated while 20 remain in their current positions, reflecting a strategic redistribution of the workforce to support expansion and operational efficiency.






